## Things to note
1. get data into pandas/numpy. Appy one-hot encoding per city. remove air_pollution_features_row_id. remove time of measurement. 
2. train test split must be chronological to prevent data leakage.
3. Use light gbm (classifier) to reduce training time. Note that classifier does not take into consideration that 1 and 2 are close predictions, so evaluation should take into consideration that (get both exact accuracy and within one accuracy scores)
4. evaluate model (also compare it against just guessing the 6 hour air pollution = current air pollution since they are highly correlated anyway. my model should outperform that) and save it to a file in minio

In [11]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
import clickhouse_connect
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
client = clickhouse_connect.get_client(
    host='localhost',
    port=8143,
    username='default',
    password=os.getenv("CLICKHOUSE_PASSWORD")
)

In [12]:
air_pollution_features_df = client.query_df("SELECT * FROM weather_features.feature_air_pollution order by time_of_measurement")
air_pollution_features_df = air_pollution_features_df.dropna()

# Appy one-hot encoding per city. remove air_pollution_features_row_id. remove time of measurement. 
air_pollution_features_df = pd.get_dummies(air_pollution_features_df, columns=['lat_and_long_encoding_id'], dtype=int)
air_pollution_features_df=air_pollution_features_df.drop(columns="air_pollution_features_row_id")
air_pollution_features_df=air_pollution_features_df.drop(columns="time_of_measurement") # relevant time features like day and hour and weekend or not are encoded already.

In [13]:
len(air_pollution_features_df)

187471

In [14]:
air_pollution_features_df.head(100)

,co_concentration,no_concentration,no2_concentration,o3_concentration,so2_concentration,pm2_5_concentration,pm10_concentration,nh3_concentration,aqi,aqi_1_hour_past,...,hour_of_day,day_of_week,is_weekend,aqi_6_hour_ahead,lat_and_long_encoding_id_46ca04f3300b447ab225f808c24c1f2a,lat_and_long_encoding_id_60ec2e136e4b7aa88a44078462ab7bf0,lat_and_long_encoding_id_69f0f0f1012c1bce555f7de22f849232,lat_and_long_encoding_id_6e26eb15061b34b2b95908f46f50c496,lat_and_long_encoding_id_dc07b90435445c5e64d7c73cbf766827,lat_and_long_encoding_id_de87d594f95f5454e6861e2026e664d0
1008,290.390015,0.000000,1.970000,50.779999,0.410000,94.059998,273.670013,1.71,5,5,...,0,7,1,5,0,1,0,0,0,0
1009,293.730011,0.000000,2.100000,57.220001,1.280000,11.880000,20.990000,6.84,2,2,...,0,7,1,2,0,0,1,0,0,0
1010,243.660004,2.490000,27.080000,0.460000,37.669998,9.110000,16.540001,1.22,2,2,...,0,7,1,2,0,0,0,0,1,0
1011,447.269989,0.000000,3.770000,32.900002,1.100000,21.410000,92.870003,2.53,3,3,...,0,7,1,3,1,0,0,0,0,0
1012,647.539978,50.509998,39.070000,0.000000,26.700001,34.369999,62.450001,3.45,3,3,...,0,7,1,4,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1103,1455.310059,6.200000,19.879999,45.060001,5.360000,23.440001,29.639999,3.64,2,2,...,15,7,1,2,0,0,0,0,0,1
1104,727.650024,4.860000,20.730000,50.070000,0.910000,38.680000,86.279999,1.38,3,3,...,16,7,1,4,0,1,0,0,0,0
1105,330.450012,0.000000,4.670000,70.099998,2.950000,14.010000,21.570000,6.02,2,2,...,16,7,1,3,0,0,1,0,0,0
1106,814.440002,19.219999,66.489998,10.460000,30.990000,25.639999,36.200001,7.41,3,2,...,16,7,1,3,0,0,0,0,1,0


In [15]:
# Count rows where you can guess the aqi 6 hours ahead because it is the same, as a comparison baseline
current_and_six_hour_ahead_same_fraction = ((air_pollution_features_df['aqi'] == air_pollution_features_df['aqi_6_hour_ahead']).sum())/len(air_pollution_features_df)
print(current_and_six_hour_ahead_same_fraction)

0.6090275295912434


In [16]:
Y=air_pollution_features_df["aqi_6_hour_ahead"]
X=air_pollution_features_df.drop(columns="aqi_6_hour_ahead")

# Slice chronologically using .iloc; 80-20 split
split_idx = int(len(air_pollution_features_df) * 0.8)
X_train,X_test = X.iloc[:split_idx],X.iloc[split_idx:]
Y_train,Y_test = Y.iloc[:split_idx],Y.iloc[split_idx:]

In [17]:
X_train

,co_concentration,no_concentration,no2_concentration,o3_concentration,so2_concentration,pm2_5_concentration,pm10_concentration,nh3_concentration,aqi,aqi_1_hour_past,...,pm10_3_hour_delta,hour_of_day,day_of_week,is_weekend,lat_and_long_encoding_id_46ca04f3300b447ab225f808c24c1f2a,lat_and_long_encoding_id_60ec2e136e4b7aa88a44078462ab7bf0,lat_and_long_encoding_id_69f0f0f1012c1bce555f7de22f849232,lat_and_long_encoding_id_6e26eb15061b34b2b95908f46f50c496,lat_and_long_encoding_id_dc07b90435445c5e64d7c73cbf766827,lat_and_long_encoding_id_de87d594f95f5454e6861e2026e664d0
1008,290.390015,0.000000,1.97,50.779999,0.410000,94.059998,273.670013,1.71,5,5,...,49.590012,0,7,1,0,1,0,0,0,0
1009,293.730011,0.000000,2.10,57.220001,1.280000,11.880000,20.990000,6.84,2,2,...,-0.450001,0,7,1,0,0,1,0,0,0
1010,243.660004,2.490000,27.08,0.460000,37.669998,9.110000,16.540001,1.22,2,2,...,-5.879999,0,7,1,0,0,0,0,1,0
1011,447.269989,0.000000,3.77,32.900002,1.100000,21.410000,92.870003,2.53,3,3,...,-2.669998,0,7,1,1,0,0,0,0,0
1012,647.539978,50.509998,39.07,0.000000,26.700001,34.369999,62.450001,3.45,3,3,...,0.860001,0,7,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155533,159.860001,0.000000,1.73,66.760002,0.540000,12.690000,27.559999,1.15,2,2,...,6.750000,18,7,1,0,0,1,0,0,0
155534,189.729996,0.000000,10.59,31.570000,3.630000,12.200000,12.900000,0.17,2,2,...,-0.190001,18,7,1,0,0,0,0,1,0
155535,132.050003,0.000000,0.09,29.629999,0.050000,1.640000,3.040000,0.14,1,1,...,0.120000,18,7,1,1,0,0,0,0,0
155536,214.580002,0.000000,17.98,59.509998,8.760000,31.139999,83.510002,4.10,3,3,...,4.090004,18,7,1,0,0,0,1,0,0


In [ ]:
model=LGBMClassifier()
model.fit(X_train,Y_train)
Y_pred=model.predict(X_test)
acs=accuracy_score(Y_test,Y_pred)
print(acs*100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044870 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10769
[LightGBM] [Info] Number of data points in the train set: 149976, number of used features: 63
[LightGBM] [Info] Start training from score -1.349712
[LightGBM] [Info] Start training from score -1.061735
[LightGBM] [Info] Start training from score -1.479098
[LightGBM] [Info] Start training from score -2.439162
[LightGBM] [Info] Start training from score -2.528824
